In [33]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler, LabelEncoder
from sklearn.impute import SimpleImputer  # Import SimpleImputer from sklearn.impute
from sklearn.pipeline import Pipeline
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from tensorflow.keras.utils import to_categorical
from scikeras.wrappers import KerasClassifier

In [34]:
# Load data
features = pd.read_csv('train_values.csv')
labels = pd.read_csv('train_labels.csv')
df = pd.concat([features, labels['damage_grade']], axis=1)

In [35]:
df.describe()

,building_id,geo_level_1_id,geo_level_2_id,geo_level_3_id,count_floors_pre_eq,age,area_percentage,height_percentage,has_superstructure_adobe_mud,has_superstructure_mud_mortar_stone,...,has_secondary_use_hotel,has_secondary_use_rental,has_secondary_use_institution,has_secondary_use_school,has_secondary_use_industry,has_secondary_use_health_post,has_secondary_use_gov_office,has_secondary_use_use_police,has_secondary_use_other,damage_grade
count,2.606010e+05,260601.000000,260601.000000,260601.000000,260601.000000,260601.000000,260601.000000,260601.000000,260601.000000,260601.000000,...,260601.000000,260601.000000,260601.000000,260601.000000,260601.000000,260601.000000,260601.000000,260601.000000,260601.000000,260601.000000
mean,5.256755e+05,13.900353,701.074685,6257.876148,2.129723,26.535029,8.018051,5.434365,0.088645,0.761935,...,0.033626,0.008101,0.000940,0.000361,0.001071,0.000188,0.000146,0.000088,0.005119,2.238272
std,3.045450e+05,8.033617,412.710734,3646.369645,0.727665,73.565937,4.392231,1.918418,0.284231,0.425900,...,0.180265,0.089638,0.030647,0.018989,0.032703,0.013711,0.012075,0.009394,0.071364,0.611814
min,4.000000e+00,0.000000,0.000000,0.000000,1.000000,0.000000,1.000000,2.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,1.000000
25%,2.611900e+05,7.000000,350.000000,3073.000000,2.000000,10.000000,5.000000,4.000000,0.000000,1.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,2.000000
50%,5.257570e+05,12.000000,702.000000,6270.000000,2.000000,15.000000,7.000000,5.000000,0.000000,1.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,2.000000
75%,7.897620e+05,21.000000,1050.000000,9412.000000,2.000000,30.000000,9.000000,6.000000,0.000000,1.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,3.000000
max,1.052934e+06,30.000000,1427.000000,12567.000000,9.000000,995.000000,100.000000,32.000000,1.000000,1.000000,...,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,3.000000


In [36]:
df.describe(include='O')

,land_surface_condition,foundation_type,roof_type,ground_floor_type,other_floor_type,position,plan_configuration,legal_ownership_status
count,260601,260601,260601,260601,260601,260601,260601,260601
unique,3,5,3,5,4,4,10,4
top,t,r,n,f,q,s,d,v
freq,216757,219196,182842,209619,165282,202090,250072,250939


In [37]:
# Split data
X_train, X_val, y_train, y_val = train_test_split(
    df.drop('damage_grade', axis=1),
    df['damage_grade'],
    random_state=20,
    shuffle=True,
    test_size=0.01
)

In [38]:
# Encode target variable
label_encoder = LabelEncoder()
y_train_encoded = label_encoder.fit_transform(y_train)
y_val_encoded = label_encoder.transform(y_val)

In [39]:
# Convert to one-hot encoded format for categorical_crossentropy
y_train_onehot = to_categorical(y_train_encoded)
y_val_onehot = to_categorical(y_val_encoded)

In [40]:
# Define columns
column_names = list(df.drop('damage_grade', axis=1).columns)
multi_class_columns = [col for col in column_names if df[col].dtype == 'object' and df[col].nunique() > 2]
binary_columns = [col for col in column_names if df[col].dtype == 'object' and df[col].nunique() == 2]
num_columns = [col for col in column_names if df[col].dtype != 'object']

In [41]:
# Define preprocessing pipelines
numeric_pipeline_simple = Pipeline([
    ('impute', SimpleImputer(strategy='median')),
    ('scale', StandardScaler())
])

In [42]:
cat_pipeline_simple = Pipeline([
    ('impute', SimpleImputer(strategy='constant', fill_value='MISSING')),
    ('encode', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

In [43]:
preprocessor_simple = ColumnTransformer([
    ('num', numeric_pipeline_simple, num_columns),
    ('cat', cat_pipeline_simple, multi_class_columns + binary_columns)
])

In [44]:
# Fit and transform the data
X_train_preprocessed = preprocessor_simple.fit_transform(X_train)
X_val_preprocessed = preprocessor_simple.transform(X_val)
# Reshape data for LSTM (samples, timesteps, features)
X_train_reshaped = X_train_preprocessed.reshape(X_train_preprocessed.shape[0], 1, X_train_preprocessed.shape[1])
X_val_reshaped = X_val_preprocessed.reshape(X_val_preprocessed.shape[0], 1, X_val_preprocessed.shape[1])

In [45]:
# Define LSTM model-building function
def build_lstm(input_dim, num_classes):
    model = Sequential()
    model.add(LSTM(64, input_shape=(1, input_dim), return_sequences=True))
    model.add(Dropout(0.2))
    model.add(LSTM(32, return_sequences=False))
    model.add(Dropout(0.2))
    model.add(Dense(32, activation='relu'))
    model.add(Dense(num_classes, activation='softmax'))
    
    model.compile(optimizer=Adam(), loss='categorical_crossentropy', metrics=['accuracy'])
    return model

# Wrap the LSTM model in SciKeras
lstm_model = KerasClassifier(
    model=build_lstm,
    input_dim=X_train_reshaped.shape[2],  # Number of features after preprocessing
    num_classes=len(label_encoder.classes_),  # Number of classes
    epochs=20,
    batch_size=256,
    callbacks=[
        EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True),
        ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=2)
    ],
    verbose=1
)
# Train the model
lstm_model.fit(
    X_train_reshaped,
    y_train_onehot,
    validation_data=(X_val_reshaped, y_val_onehot)
)

Epoch 1/20


/home/sbaniya/.local/lib/python3.11/site-packages/keras/src/layers/rnn/rnn.py:200: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


1008/1008 ━━━━━━━━━━━━━━━━━━━━ 8s 4ms/step - accuracy: 0.5817 - loss: 0.8397 - val_accuracy: 0.6260 - val_loss: 0.7577 - learning_rate: 0.0010
Epoch 2/20
1008/1008 ━━━━━━━━━━━━━━━━━━━━ 4s 4ms/step - accuracy: 0.6229 - loss: 0.7640 - val_accuracy: 0.6356 - val_loss: 0.7458 - learning_rate: 0.0010
Epoch 3/20
1008/1008 ━━━━━━━━━━━━━━━━━━━━ 4s 4ms/step - accuracy: 0.6324 - loss: 0.7531 - val_accuracy: 0.6421 - val_loss: 0.7382 - learning_rate: 0.0010
Epoch 4/20
1008/1008 ━━━━━━━━━━━━━━━━━━━━ 4s 4ms/step - accuracy: 0.6402 - loss: 0.7436 - val_accuracy: 0.6529 - val_loss: 0.7255 - learning_rate: 0.0010
Epoch 5/20
1008/1008 ━━━━━━━━━━━━━━━━━━━━ 4s 4ms/step - accuracy: 0.6490 - loss: 0.7324 - val_accuracy: 0.6624 - val_loss: 0.7049 - learning_rate: 0.0010
Epoch 6/20
1008/1008 ━━━━━━━━━━━━━━━━━━━━ 4s 4ms/step - accuracy: 0.6563 - loss: 0.7186 - val_accuracy: 0.6694 - val_loss: 0.6981 - learning_rate: 0.0010
Epoch 7/20
1008/1008 ━━━━━━━━━━━━━━━━━━━━ 4s 4ms/step - accuracy: 0.6591 - loss: 0.7147

KerasClassifier(
	model=<function build_lstm at 0x14a097534900>
	build_fn=None
	warm_start=False
	random_state=None
	optimizer=rmsprop
	loss=None
	metrics=None
	batch_size=256
	validation_batch_size=None
	verbose=1
	callbacks=[<keras.src.callbacks.early_stopping.EarlyStopping object at 0x14a0a19ed210>, <keras.src.callbacks.reduce_lr_on_plateau.ReduceLROnPlateau object at 0x14a097d294d0>]
	validation_split=0.0
	shuffle=True
	run_eagerly=False
	epochs=20
	input_dim=69
	num_classes=3
	class_weight=None
)

In [54]:
# Step 9: Evaluate the model on validation set
# Since KerasClassifier.score returns accuracy, compute it directly
val_accuracy = lstm_model.score(X_val_reshaped, y_val_onehot)
print(f"Validation Accuracy: {val_accuracy*100:.2f}%")

11/11 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step
Validation Accuracy: 68.70%


In [55]:
# Predict on validation set
y_pred_onehot = lstm_model.predict(X_val_reshaped)  # Returns probabilities or one-hot encoded
y_pred_encoded = np.argmax(y_pred_onehot, axis=1)  # Convert to class indices
y_pred = label_encoder.inverse_transform(y_pred_encoded)  # Convert back to original labels

11/11 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step 


In [56]:
# Evaluate predictions
from sklearn.metrics import accuracy_score
print("Accuracy on validation set:", accuracy_score(y_val, y_pred))
print("\nClassification Report:\n", classification_report(y_val, y_pred))

Accuracy on validation set: 0.18143459915611815

Classification Report:
               precision    recall  f1-score   support

           0       0.00      0.00      0.00       246
           1       0.33      0.04      0.06      1490
           2       0.23      0.48      0.32       871
           3       0.00      0.00      0.00         0

    accuracy                           0.18      2607
   macro avg       0.14      0.13      0.09      2607
weighted avg       0.26      0.18      0.14      2607



/opt/apps/software/lang/Anaconda3/2024.02-1/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1509: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/opt/apps/software/lang/Anaconda3/2024.02-1/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1509: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/opt/apps/software/lang/Anaconda3/2024.02-1/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1509: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, 

In [57]:
# Step 10: Predict on test set for submission
df_test = pd.read_csv('test_values.csv')
ids = df_test['building_id'].values

In [58]:
# Preprocess test data
X_test_preprocessed = preprocessor_simple.transform(df_test)  # Use transform, not fit_transform
X_test_reshaped = X_test_preprocessed.reshape(X_test_preprocessed.shape[0], 1, X_test_preprocessed.shape[1])

In [59]:
# Predict on test set
test_preds_onehot = lstm_model.predict(X_test_reshaped)  # Returns probabilities
test_preds_encoded = np.argmax(test_preds_onehot, axis=1)  # Convert to class indices
test_preds = label_encoder.inverse_transform(test_preds_encoded)  # Convert back to original labels

340/340 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step


In [60]:
# Create submission file
submission = pd.DataFrame({'building_id': ids, 'damage_grade': test_preds})
submission.to_csv('LSTM_simple_submission.csv', index=False)
print("Submission file 'LSTM_simple_submission.csv' created successfully!")

Submission file 'LSTM_simple_submission.csv' created successfully!


**Preprocessed LSTM**

In [61]:
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.preprocessing import PowerTransformer
from sklearn.preprocessing import RobustScaler
from sklearn.preprocessing import FunctionTransformer
import matplotlib.pyplot as plt

# Custom OutlierCapper
class OutlierCapper(BaseEstimator, TransformerMixin):
    def __init__(self, cols, lower=0.01, upper=0.99):
        self.cols = cols
        self.lower = lower
        self.upper = upper

    def fit(self, X, y=None):
        self.col_indices_ = [num_columns.index(c) for c in self.cols]
        sub = X[:, self.col_indices_]
        q = np.quantile(sub, [self.lower, self.upper], axis=0)
        self.lo_, self.hi_ = q[0], q[1]
        return self

    def transform(self, X):
        X = X.copy()
        for idx, lo, hi in zip(self.col_indices_, self.lo_, self.hi_):
            X[:, idx] = np.clip(X[:, idx], lo, hi)
        return X

In [62]:
# Define preprocessing pipelines
numeric_pipeline = Pipeline([
    ('impute', SimpleImputer(strategy='median')),
    ('cap', OutlierCapper(cols=['area_percentage', 'height_percentage'], lower=0.01, upper=0.99)),
    ('power', PowerTransformer(method='yeo-johnson')),
    ('scale', RobustScaler()),
])

cat_pipeline = Pipeline([
    ('impute', SimpleImputer(strategy='constant', fill_value='MISSING')),
    ('encode', OneHotEncoder(handle_unknown='ignore', sparse_output=False)),
])

# Assemble ColumnTransformer
preprocessor_complex = ColumnTransformer([
    ('num', numeric_pipeline, num_columns),
    ('cat', cat_pipeline, multi_class_columns + binary_columns),
])

In [63]:
# Get feature dimension after preprocessing
X_tr_enc2 = preprocessor_complex.fit_transform(X_train)
fd2 = X_tr_enc2.shape[1]

# Define LSTM model-building function
def build_lstm(input_dim, num_classes):
    model = Sequential()
    model.add(LSTM(64, input_shape=(1, input_dim), return_sequences=True))
    model.add(Dropout(0.2))
    model.add(LSTM(32, return_sequences=False))
    model.add(Dropout(0.2))
    model.add(Dense(32, activation='relu'))
    model.add(Dense(num_classes, activation='softmax'))
    
    model.compile(optimizer=Adam(), loss='categorical_crossentropy', metrics=['accuracy'])
    return model

# Wrap the LSTM model in SciKeras
lstm_complex = KerasClassifier(
    model=build_lstm,
    input_dim=fd2,  # Feature dimension after preprocessing
    num_classes=len(label_encoder.classes_),
    epochs=30,
    batch_size=128,
    validation_split=0.2,
    callbacks=[
        EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True),
        ReduceLROnPlateau(monitor='val_loss', factor=0.3, patience=3)
    ],
    verbose=1
)

In [64]:
# Define a reshaping transformer for LSTM
def reshape_for_lstm(X):
    return X.reshape(X.shape[0], 1, X.shape[1])

reshape_transformer = FunctionTransformer(reshape_for_lstm)

# Create the pipeline
pipe_lstm_complex = Pipeline([
    ('preproc', preprocessor_complex),
    ('reshape', reshape_transformer),
    ('clf', lstm_complex)
])

# Train the pipeline
pipe_lstm_complex.fit(X_train, y_train_onehot)

Epoch 1/30


/home/sbaniya/.local/lib/python3.11/site-packages/keras/src/layers/rnn/rnn.py:200: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


1613/1613 ━━━━━━━━━━━━━━━━━━━━ 10s 4ms/step - accuracy: 0.5845 - loss: 0.8235 - val_accuracy: 0.6218 - val_loss: 0.7588 - learning_rate: 0.0010
Epoch 2/30
1613/1613 ━━━━━━━━━━━━━━━━━━━━ 6s 3ms/step - accuracy: 0.6223 - loss: 0.7614 - val_accuracy: 0.6342 - val_loss: 0.7489 - learning_rate: 0.0010
Epoch 3/30
1613/1613 ━━━━━━━━━━━━━━━━━━━━ 6s 3ms/step - accuracy: 0.6288 - loss: 0.7543 - val_accuracy: 0.6392 - val_loss: 0.7412 - learning_rate: 0.0010
Epoch 4/30
1613/1613 ━━━━━━━━━━━━━━━━━━━━ 5s 3ms/step - accuracy: 0.6377 - loss: 0.7475 - val_accuracy: 0.6387 - val_loss: 0.7368 - learning_rate: 0.0010
Epoch 5/30
1613/1613 ━━━━━━━━━━━━━━━━━━━━ 5s 3ms/step - accuracy: 0.6407 - loss: 0.7421 - val_accuracy: 0.6477 - val_loss: 0.7268 - learning_rate: 0.0010
Epoch 6/30
1613/1613 ━━━━━━━━━━━━━━━━━━━━ 5s 3ms/step - accuracy: 0.6465 - loss: 0.7337 - val_accuracy: 0.6587 - val_loss: 0.7156 - learning_rate: 0.0010
Epoch 7/30
1613/1613 ━━━━━━━━━━━━━━━━━━━━ 5s 3ms/step - accuracy: 0.6520 - loss: 0.724

Pipeline(steps=[('preproc',
                 ColumnTransformer(transformers=[('num',
                                                  Pipeline(steps=[('impute',
                                                                   SimpleImputer(strategy='median')),
                                                                  ('cap',
                                                                   OutlierCapper(cols=['area_percentage',
                                                                                       'height_percentage'])),
                                                                  ('power',
                                                                   PowerTransformer()),
                                                                  ('scale',
                                                                   RobustScaler())]),
                                                  ['building_id',
                                                   'geo_level_1_id',
                                                   'geo_level_2_id',
                                                   'geo_level_3_id',
                                                   'count_floors_pre_eq', 'age',
                                                   'area_...
                 FunctionTransformer(func=<function reshape_for_lstm at 0x14a09ff98e00>)),
                ('clf',
                 KerasClassifier(batch_size=128, callbacks=[<keras.src.callbacks.early_stopping.EarlyStopping object at 0x14a09f2e06d0>, <keras.src.callbacks.reduce_lr_on_plateau.ReduceLROnPlateau object at 0x14a09ecd7d50>], epochs=30, input_dim=69, model=<function build_lstm at 0x14a09f8e3ec0>, num_classes=3, validation_split=0.2))])

In [65]:
# Evaluate on validation set
val_accuracy = pipe_lstm_complex.score(X_val, y_val_onehot)
print(f"Validation Accuracy: {val_accuracy*100:.2f}%")

# Predict on validation set
y_pred_onehot = pipe_lstm_complex.predict(X_val)
y_pred_encoded = np.argmax(y_pred_onehot, axis=1)
y_pred = label_encoder.inverse_transform(y_pred_encoded)

# Evaluate predictions
# Evaluate predictions
print("Accuracy on validation set:", accuracy_score(y_val, y_pred))
print("F1-micro score:", f1_score(y_val, y_pred, average='micro'))
print("F1-macro score:", f1_score(y_val, y_pred, average='macro'))
print("F1-weighted score:", f1_score(y_val, y_pred, average='weighted'))
print("\nClassification Report:\n", classification_report(y_val, y_pred))

21/21 ━━━━━━━━━━━━━━━━━━━━ 1s 16ms/step
Validation Accuracy: 68.51%
21/21 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step 
Accuracy on validation set: 0.18795550441120062
F1-micro score: 0.18795550441120062
F1-macro score: 0.09460464486582106
F1-weighted score: 0.13806181002852466

Classification Report:
               precision    recall  f1-score   support

           0       0.00      0.00      0.00       246
           1       0.28      0.03      0.05      1490
           2       0.24      0.52      0.33       871
           3       0.00      0.00      0.00         0

    accuracy                           0.19      2607
   macro avg       0.13      0.14      0.09      2607
weighted avg       0.24      0.19      0.14      2607



/opt/apps/software/lang/Anaconda3/2024.02-1/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1509: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/opt/apps/software/lang/Anaconda3/2024.02-1/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1509: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/opt/apps/software/lang/Anaconda3/2024.02-1/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1509: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, 

In [66]:
# Predict on test set for submission
df_test = pd.read_csv('test_values.csv')
ids = df_test['building_id'].values

# Predict on test set
test_preds_onehot = pipe_lstm_complex.predict(df_test)
test_preds_encoded = np.argmax(test_preds_onehot, axis=1)
test_preds = label_encoder.inverse_transform(test_preds_encoded)

679/679 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step


In [67]:
# Create submission file
submission = pd.DataFrame({'building_id': ids, 'damage_grade': test_preds})
submission.to_csv('LSTM_complex_submission.csv', index=False)
print("Submission file 'LSTM_complex_submission.csv' created successfully!")

Submission file 'LSTM_complex_submission.csv' created successfully!


**Hyperparameter Tuning**

In [53]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler, LabelEncoder, PowerTransformer, RobustScaler
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.metrics import f1_score
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout, BatchNormalization, Input
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from tensorflow.keras.metrics import Metric
import tensorflow as tf
from keras_tuner import HyperModel, RandomSearch, Objective

# Custom Micro-F1 Metric for Keras
class MicroF1Metric(Metric):
    def __init__(self, name='micro_f1', **kwargs):
        super().__init__(name=name, **kwargs)
        self.true_positives = self.add_weight(name='tp', initializer='zeros')
        self.false_positives = self.add_weight(name='fp', initializer='zeros')
        self.false_negatives = self.add_weight(name='fn', initializer='zeros')

    def update_state(self, y_true, y_pred, sample_weight=None):
        y_true = tf.cast(y_true, tf.int32)
        y_pred = tf.argmax(y_pred, axis=1)
        y_pred = tf.cast(y_pred, tf.int32)
        
        true_pos = tf.reduce_sum(tf.cast(tf.equal(y_true, y_pred), tf.float32) * tf.cast(y_true, tf.float32))
        false_pos = tf.reduce_sum(tf.cast(tf.not_equal(y_true, y_pred), tf.float32) * tf.cast(y_pred, tf.float32))
        false_neg = tf.reduce_sum(tf.cast(tf.not_equal(y_true, y_pred), tf.float32) * tf.cast(y_true, tf.float32))
        
        self.true_positives.assign_add(true_pos)
        self.false_positives.assign_add(false_pos)
        self.false_negatives.assign_add(false_neg)

    def result(self):
        precision = self.true_positives / (self.true_positives + self.false_positives + tf.keras.backend.epsilon())
        recall = self.true_positives / (self.true_positives + self.false_negatives + tf.keras.backend.epsilon())
        micro_f1 = 2 * (precision * recall) / (precision + recall + tf.keras.backend.epsilon())
        return micro_f1

    def reset_states(self):
        self.true_positives.assign(0.0)
        self.false_positives.assign(0.0)
        self.false_negatives.assign(0.0)

# Load training data
features = pd.read_csv('train_values.csv')
labels = pd.read_csv('train_labels.csv')
df = pd.concat([features, labels['damage_grade']], axis=1)

# Split data
X_train, X_val, y_train, y_val = train_test_split(
    df.drop('damage_grade', axis=1),
    df['damage_grade'],
    random_state=20,
    shuffle=True,
    test_size=0.01
)

# Encode labels to [0, 1, 2]
label_encoder = LabelEncoder()
y_train = label_encoder.fit_transform(y_train)
y_val = label_encoder.transform(y_val)

# Verify label range
print("Unique labels in y_train:", np.unique(y_train))
print("Unique labels in y_val:", np.unique(y_val))
if np.any(y_train > 2) or np.any(y_val > 2):
    raise ValueError("Labels contain values outside [0, 2]:", np.unique(y_train), np.unique(y_val))

# Define columns
column_names = list(df.drop('damage_grade', axis=1).columns)
multi_class_columns = [col for col in column_names if df[col].dtype == 'object' and df[col].nunique() > 2]
binary_columns = [col for col in column_names if df[col].dtype == 'object' and df[col].nunique() == 2]
num_columns = [col for col in column_names if df[col].dtype != 'object']

# Custom OutlierCapper
class OutlierCapper(BaseEstimator, TransformerMixin):
    def __init__(self, cols, lower=0.01, upper=0.99):
        self.cols = cols
        self.lower = lower
        self.upper = upper

    def fit(self, X, y=None):
        self.col_indices_ = [num_columns.index(c) for c in self.cols]
        sub = X[:, self.col_indices_]
        q = np.quantile(sub, [self.lower, self.upper], axis=0)
        self.lo_, self.hi_ = q[0], q[1]
        return self

    def transform(self, X):
        X = X.copy()
        for idx, lo, hi in zip(self.col_indices_, self.lo_, self.hi_):
            X[:, idx] = np.clip(X[:, idx], lo, hi)
        return X

# Define preprocessing pipelines
numeric_pipeline = Pipeline([
    ('impute', SimpleImputer(strategy='median')),
    ('cap', OutlierCapper(cols=['area_percentage', 'height_percentage'], lower=0.01, upper=0.99)),
    ('power', PowerTransformer(method='yeo-johnson')),
    ('scale', RobustScaler()),
])

cat_pipeline = Pipeline([
    ('impute', SimpleImputer(strategy='constant', fill_value='MISSING')),
    ('encode', OneHotEncoder(handle_unknown='ignore', sparse_output=False)),
])

# Assemble ColumnTransformer
preprocessor_complex = ColumnTransformer([
    ('num', numeric_pipeline, num_columns),
    ('cat', cat_pipeline, multi_class_columns + binary_columns),
])

# Preprocess training and validation data
X_tr_enc2 = preprocessor_complex.fit_transform(X_train)
X_val_enc2 = preprocessor_complex.transform(X_val)

# Get feature dimension
feature_dim = X_tr_enc2.shape[1]

# Reshape data for LSTM (samples, timesteps, features)
X_tr_enc2_reshaped = X_tr_enc2.reshape(X_tr_enc2.shape[0], 1, X_tr_enc2.shape[1])
X_val_enc2_reshaped = X_val_enc2.reshape(X_val_enc2.shape[0], 1, X_tr_enc2.shape[1])

# Define LSTM HyperModel
class LSTMHyper(HyperModel):
    def build(self, hp):
        model = Sequential([Input(shape=(1, feature_dim))])
        for i in range(hp.Int('n_layers', 1, 3)):
            units = hp.Int(f'units_{i}', 32, 256, step=32)
            return_sequences = i < hp.Int('n_layers', 1, 3) - 1  # Only last LSTM layer should not return sequences
            model.add(LSTM(units, return_sequences=return_sequences))
            if hp.Boolean(f'use_bn_{i}'):
                model.add(BatchNormalization())
            model.add(Dropout(hp.Float(f'dropout_{i}', 0, 0.5, step=0.1)))
        model.add(Dense(32, activation='relu'))
        model.add(Dense(3, activation='softmax'))
        model.compile(
            optimizer=Adam(hp.Float('lr', 1e-4, 1e-2, sampling='log')),
            loss='sparse_categorical_crossentropy',
            metrics=['accuracy', MicroF1Metric()]
        )
        return model

# Create tuner
tuner = RandomSearch(
    LSTMHyper(),
    objective=Objective('val_micro_f1', direction='max'),
    max_trials=10,
    executions_per_trial=1,
    directory='lstm_tune',
    project_name='lstm2'
)

# Tuning
tuner.search(
    X_tr_enc2_reshaped, y_train,
    validation_data=(X_val_enc2_reshaped, y_val),
    epochs=20,
    callbacks=[
        EarlyStopping(monitor='val_loss', patience=4),
        ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=2)
    ]
)

# Best Model
best_lstm = tuner.get_best_models(1)[0]

# Evaluate best model on validation set
val_loss, val_accuracy, val_micro_f1 = best_lstm.evaluate(X_val_enc2_reshaped, y_val)
print(f"Best LSTM Validation Accuracy: {val_accuracy*100:.2f}%")
print(f"Best LSTM Validation Micro-F1: {val_micro_f1:.4f}")

# Predict on validation set
y_pred_probs = best_lstm.predict(X_val_enc2_reshaped)
y_pred = np.argmax(y_pred_probs, axis=1)
y_pred_labels = y_pred + 1  # Shift back to original labels [1, 2, 3]

# Evaluate predictions with micro-F1
val_micro_f1_sklearn = f1_score(y_val + 1, y_pred_labels, average='micro')
print(f"Validation Micro-F1 (sklearn): {val_micro_f1_sklearn:.4f}")

# Debug: Verify micro-F1 calculation
print("Sample of true labels:", (y_val + 1)[:10])
print("Sample of predicted labels:", y_pred_labels[:10])

# Predict on test set for submission
df_test = pd.read_csv('test_values.csv')
ids = df_test['building_id'].values

# Preprocess test data
X_test_enc = preprocessor_complex.transform(df_test)
X_test_enc_reshaped = X_test_enc.reshape(X_test_enc.shape[0], 1, X_test_enc.shape[1])

# Predict on test set
test_preds_probs = best_lstm.predict(X_test_enc_reshaped)
test_preds = np.argmax(test_preds_probs, axis=1)
test_preds_labels = test_preds + 1  # Shift back to original labels [1, 2, 3]

# Create submission file
submission = pd.DataFrame({'building_id': ids, 'damage_grade': test_preds_labels})
submission.to_csv('LSTM_tuned_submission.csv', index=False)
print("Submission file 'LSTM_tuned_submission.csv' created successfully!")

Unique labels in y_train: [0 1 2]
Unique labels in y_val: [0 1 2]
Reloading Tuner from lstm_tune/lstm2/tuner0.json


/home/sbaniya/.local/lib/python3.11/site-packages/keras/src/saving/saving_lib.py:757: UserWarning: Skipping variable loading for optimizer 'adam', because it has 2 variables whereas the saved optimizer has 26 variables. 
  saveable.load_own_variables(weights_store.get(inner_path))


82/82 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.6974 - loss: 0.6468 - micro_f1: 0.6804 
Best LSTM Validation Accuracy: 70.54%
Best LSTM Validation Micro-F1: 0.6918
82/82 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step
Validation Micro-F1 (sklearn): 0.7054
Sample of true labels: [3 2 2 2 3 2 2 2 1 2]
Sample of predicted labels: [2 2 2 1 2 2 2 2 2 2]
2715/2715 ━━━━━━━━━━━━━━━━━━━━ 3s 1ms/step
Submission file 'LSTM_tuned_submission.csv' created successfully!
